<a href="https://colab.research.google.com/github/marcocslima/rag_tests/blob/main/Llamaindex_PropertyGraphIndex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from IPython.display import clear_output
# 1. Instalação das bibliotecas de integração
# Descomente a linha do provedor que deseja usar:
!pip install -q llama-index llama-index-llms-groq         # Para Groq
!pip install -q llama-index llama-index-llms-openrouter  # Para OpenRouter

#!pip install -q -U llama-index-embeddings-huggingface
!pip install -q llama-index-embeddings-huggingface

clear_output()

In [ ]:
import os
import time
from google.colab import userdata
from llama_index.core import PropertyGraphIndex, SimpleDirectoryReader, Settings
from llama_index.core.indices.property_graph import DynamicLLMPathExtractor
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq

import nest_asyncio
nest_asyncio.apply()

In [ ]:
# --- ESCOLHA SEU PROVEDOR ABAIXO ---

# OPÇÃO A: GROQ
from llama_index.llms.groq import Groq
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY_MCL')
llm = Groq(model="llama-3.3-70b-versatile") # Modelo potente para extração jurídica

# OPÇÃO B: OPENROUTER
# from llama_index.llms.openrouter import OpenRouter
# os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
# llm = OpenRouter(model="meta-llama/llama-3-70b-instruct") # Exemplo de modelo

# Configuração do Modelo de Embedding Local (Gratuito)
# Isso evita que o sistema peça uma chave da OpenAI.
#embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
#embed_model = HuggingFaceEmbedding(model_name="intfloat/multilingual-e5-large")
#embed_model = HuggingFaceEmbedding(model_name="intfloat/multilingual-e5-small")
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")

# Aplicando as configurações globalmente no Settings
Settings.llm = llm
Settings.embed_model = embed_model

clear_output()

In [ ]:
# 2. Preparação e Ingestão (Mesma lógica anterior)
if not os.path.exists("data"): os.makedirs("data")
# ... (adicione seus documentos na pasta 'data')
documents = SimpleDirectoryReader("/content/drive/MyDrive/Tmp/RAG_DOCS/Decretos").load_data()

In [ ]:
# Entidades baseadas na estrutura normativa e conceitos das fontes [1-3]
entities = [
    "NORMA",            # Ex: Lei nº 5.172/1966 [4, 5]
    "ESTRUTURA",        # Ex: Livro, Título, Capítulo, Artigo [1, 3]
    "TRIBUTO",          # Ex: Impostos, Taxas, Contribuições [2, 6]
    "IMPOSTO_ESPECIFICO",# Ex: IPTU [7, 8], ITBI [9, 10], ISSQN [11]
    "FATO_GERADOR",     # Ex: Situação necessária à obrigação [12, 13]
    "BASE_DE_CALCULO",  # Ex: Valor venal ou preço do serviço [14-17]
    "ALIQUOTA",         # Ex: Percentual aplicado sobre a base [14, 18]
    "SUJEITO_ATIVO",    # Ex: Pessoa jurídica de direito público [19]
    "SUJEITO_PASSIVO",  # Ex: Contribuinte ou responsável [19, 20]
    "CREDITO_TRIBUTARIO",# Ex: Valor devido após lançamento [21, 22]
    "PROCEDIMENTO",     # Ex: Lançamento, Fiscalização, Consulta [23-26]
    "PENALIDADE",       # Ex: Multas e juros de mora [27, 28]
    "DOCUMENTO_FISCAL"  # Ex: Nota Fiscal, Certidão Negativa [29-32]
]

# Relações baseadas na lógica tributária das fontes [5, 33-35]
relations = [
    "CONTEM",           # Hierarquia documental [1, 3]
    "REGULA",           # A Lei regula o sistema tributário [5]
    "INCIDE_SOBRE",     # Tributo incide sobre propriedade ou serviço [7, 8, 11]
    "TEM_COMO_FATO_GERADOR", # Definição legal do fato gerador [2, 12]
    "DETERMINA",        # Lei determina base e alíquota [14]
    "NOTIFICA",         # Lançamento notificado ao sujeito passivo [36, 37]
    "SUSPENDE",         # Causas de suspensão do crédito [33, 38]
    "EXTINGUE",         # Causas de extinção (ex: pagamento) [34, 39]
    "EXCLUI",           # Causas de exclusão (ex: isenção, anistia) [35, 40]
    "VINCULA_SE_A",     # Obrigação acessória vinculada à principal [41, 42]
    "CITA",             # Artigo ou Norma citando outro dispositivo [5, 43]
    "APLICA_PENALIDADE" # Infração resulta em penalidade [23, 44]
]

# Use estes nomes de argumentos conforme a documentação mais recente (v0.12+)
kg_extractor = DynamicLLMPathExtractor(
    llm=llm,
    allowed_entity_types=entities,
    allowed_relation_types=relations,
    num_workers=1 # Processa um por um para evitar estourar o limite de tokens por minuto
)

In [ ]:
# 4. Construção do Índice Híbrido (Texto + Grafo)
# Essa abordagem híbrida é a mais recomendada para documentos jurídicos [4, 5].
index = PropertyGraphIndex.from_documents(
    documents,
    kg_extractors=[kg_extractor],
    show_progress=True
)

clear_output()

In [ ]:
# Defina seu extractor com poucos workers para reduzir paralelismo
kg_extractor = DynamicLLMPathExtractor(
    llm=llm,
    allowed_entity_types=entities,
    allowed_relation_types=relations,
    max_triplets_per_chunk=10,  # Reduza se necessário
    num_workers=1               # Apenas 1 worker para evitar múltiplas requisições simultâneas
)

# Função para inserir documentos com delay entre cada inserção
def insert_with_delay(documents, delay=10):
    index = None
    for i, doc in enumerate(documents):
        if i == 0:
            index = PropertyGraphIndex.from_documents(
                [doc],
                kg_extractors=[kg_extractor],
                show_progress=True
            )
        else:
            index.insert(doc)
        print(f"Documento (i+1) processado. Aguardando {delay}s para o próximo...")
        time.sleep(delay)
    return index

# Use a função para criar o índice
index = insert_with_delay(documents, delay=10)
